# Kenya Climate EDA
Task 2 notebook for Kenya (profiling, cleaning, time-series analysis, correlation, and distribution analysis).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore

In [ ]:
country = 'Kenya'
df = pd.read_csv('../data/kenya.csv')
df['Country'] = country
df['DATE'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j')
df['Month'] = df['DATE'].dt.month
df = df.replace(-999, np.nan)
dup_count = int(df.duplicated().sum())
df = df.drop_duplicates()
print('Duplicate rows found and removed:', dup_count)

In [ ]:
display(df.describe(include='number'))
missing = df.isna().sum().to_frame('missing_count')
missing['missing_pct'] = (missing['missing_count'] / len(df)) * 100
display(missing.sort_values('missing_pct', ascending=False))

## Interpretation Notes
- Add a brief interpretation of key summary stats.
- List columns with >5% missing and explain analysis implications.

In [ ]:
z_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX']
z_input = df[z_cols].copy().fillna(df[z_cols].median(numeric_only=True))
z_abs = np.abs(zscore(z_input, nan_policy='omit'))
outlier_rows = (z_abs > 3).any(axis=1)
print('Outlier rows (|Z| > 3):', int(outlier_rows.sum()))

## Outlier Decision
Document whether you retain, cap, or drop outliers and why.

In [ ]:
row_missing_ratio = df.isna().mean(axis=1)
df = df[row_missing_ratio <= 0.30].copy()
weather_cols = ['T2M','T2M_MAX','T2M_MIN','T2M_RANGE','PRECTOTCORR','RH2M','WS2M','WS2M_MAX','PS','QV2M']
for col in weather_cols:
    if col in df.columns:
        df[col] = df[col].ffill()
df.to_csv('../data/kenya_clean.csv', index=False)
print('Saved cleaned file to ../data/kenya_clean.csv')

In [ ]:
monthly_temp = df.resample('M', on='DATE')['T2M'].mean()
warmest = monthly_temp.idxmax()
coolest = monthly_temp.idxmin()
plt.figure(figsize=(12, 4))
monthly_temp.plot()
plt.scatter([warmest, coolest], [monthly_temp.max(), monthly_temp.min()], color=['red', 'blue'])
plt.title('Monthly Average T2M - Kenya')
plt.ylabel('Temperature (°C)')
plt.show()

In [ ]:
monthly_prec = df.resample('M', on='DATE')['PRECTOTCORR'].sum()
peak_idx = monthly_prec.sort_values(ascending=False).head(3).index
plt.figure(figsize=(12, 4))
monthly_prec.plot(kind='bar')
for i in peak_idx:
    plt.axvline(i, color='orange', alpha=0.2)
plt.title('Monthly Total PRECTOTCORR - Kenya')
plt.ylabel('Precipitation (mm/month)')
plt.tight_layout()
plt.show()

## Time-Series Interpretation
Describe visible trends, anomalies, and likely peak rainy seasons.

In [ ]:
corr = df.select_dtypes(include='number').corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Correlation Heatmap - Kenya')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=df, x='T2M', y='RH2M', ax=axes[0])
axes[0].set_title('T2M vs RH2M')
sns.scatterplot(data=df, x='T2M_RANGE', y='WS2M', ax=axes[1])
axes[1].set_title('T2M_RANGE vs WS2M')
plt.tight_layout()
plt.show()

## Correlation Interpretation
Identify and explain the three strongest correlations.

In [ ]:
plt.figure(figsize=(10, 4))
sns.histplot(df['PRECTOTCORR'], bins=50)
plt.yscale('log')
plt.title('PRECTOTCORR Distribution (log-y) - Kenya')
plt.show()

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='T2M', y='RH2M', size='PRECTOTCORR', sizes=(10, 200), alpha=0.5)
plt.title('Bubble: T2M vs RH2M (size = PRECTOTCORR) - Kenya')
plt.show()